# GAD-NR: Graph Anomaly Detection via Neighborhood Reconstruction
## Reproducing Tables 2, 3, and 4 — Weibo Dataset

### Corrections vs prior attempt
1. **Bug fixed**: Lambda scheduling (`lambda_loss2 += 0.5`, `lambda_loss3 /= 2` every `loss_step` epochs) was
   **completely missing** from the train loop — this is the primary cause of ~72% AUC instead of ~87%.
2. **Bug fixed**: Hyperparameter values now match the paper: λ_n=0.001 (`lambda_loss1`), λ_x=0.8 (`lambda_loss2`),
   λ_d=0.5 (`lambda_loss3`). Previous code had λ_x and λ_d swapped and λ_n 10× too large.
3. **Bug fixed**: `loss_step=100` (original default) instead of 5000 which blocked scheduling.
4. Ablation configs corrected to use the right base values before zeroing out each component.

In [1]:
import sys
import os
import platform
import subprocess
import time
import math
import random
import statistics
import argparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
import networkx as nx
import scipy
import scipy.optimize
from scipy.linalg import sqrtm
from scipy.optimize import linear_sum_assignment
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
from torch.autograd import Variable
from torch.utils.data import Dataset
!pip install torch_geometric
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GINConv, SAGEConv, GATConv, PNAConv, GraphSAGE
from torch_geometric.utils import add_self_loops
from torch_geometric.transforms import normalize_features
!pip install pygod
from pygod.utils import load_data
from pygod.utils.utility import check_parameter

# Handle PyGOD metric and generator changes across versions
try:
    from pygod.metrics import eval_roc_auc
except ImportError:
    from sklearn.metrics import roc_auc_score
    def eval_roc_auc(label, score):
        return roc_auc_score(label, score)

try:
    from pygod.generator import gen_contextual_outliers, gen_structural_outliers
except ImportError:
    from pygod.generator import gen_contextual_outlier as _gen_contextual_outlier
    from pygod.generator import gen_structural_outlier as _gen_structural_outlier
    def gen_contextual_outliers(data, n, k, random_state=None):
        return _gen_contextual_outlier(data=data, n=n, k=k, seed=random_state)
    def gen_structural_outliers(data, m, n, p=0, random_state=None):
        return _gen_structural_outlier(data=data, m=m, n=n, p=p, seed=random_state)

# PyTorch >= 2.6 requires explicitly allowing PyG storage deserialization
try:
    from torch_geometric.data.storage import GlobalStorage
    torch.serialization.add_safe_globals([GlobalStorage])
except Exception:
    pass

# Environment Handling
IS_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
IS_COLAB = 'google.colab' in sys.modules
IS_APPLE_SILICON = platform.system() == 'Darwin' and platform.machine() == 'arm64'

if IS_APPLE_SILICON:
    device = torch.device('cpu') # Force CPU on Apple Silicon to prevent PyG MPS bugs
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Environment: Colab={IS_COLAB}, Kaggle={IS_KAGGLE}, Apple Silicon={IS_APPLE_SILICON}")
print(f"Using device: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.3 MB/s eta 0:00:00
Environment: Colab=True, Kaggle=False, Apple Silicon=False
Using device: cuda


## Global Args

In [2]:
class Args:
    pass

args = Args()

# Scoring weights for adaptive anomaly score (Eq. 8 in paper)
# These are lambda-prime values used ONLY for scoring, not training
args.real_loss         = False   # Use adaptive combined score (not raw training loss)
args.neigh_loss        = 'KL'
args.h_loss_weight     = 1.0     # lambda'_n for scoring
args.feature_loss_weight = 2.0   # lambda'_x for scoring (higher = more sensitive to contextual)
args.degree_loss_weight  = 1.0   # lambda'_d for scoring
args.plot_loss         = False
args.normalize_feat    = False
args.aggregator        = 'mean'
args.use_combine_outlier = False

print('Args configured for Weibo.')

Args configured for Weibo.


## Utility Functions

In [3]:
def gen_joint_structural_outliers(data, m, n, random_state=None):
    if not isinstance(data, Data):
        raise TypeError('data should be torch_geometric.data.Data')
    check_parameter(m, low=0, high=data.num_nodes, param_name='m')
    check_parameter(n, low=0, high=data.num_nodes, param_name='n')
    check_parameter(m * n, low=0, high=data.num_nodes, param_name='m*n')
    if random_state:
        np.random.seed(random_state)
    outlier_idx = np.random.choice(data.num_nodes, size=n, replace=False)
    new_edges = []
    for i in range(n):
        other_idx = np.random.choice(data.num_nodes, size=m, replace=False)
        for j in other_idx:
            new_edges.append(torch.tensor([[outlier_idx[i], j]], dtype=torch.long))
    new_edges = torch.cat(new_edges)
    y_outlier = torch.zeros(data.x.shape[0], dtype=torch.long)
    y_outlier[outlier_idx] = 1
    data.edge_index = torch.cat([data.edge_index, new_edges.T], dim=1)
    return data, y_outlier


def sanitize_score_tensor(values, name='tensor'):
    values = values.reshape(-1).clone().float()
    finite_mask = torch.isfinite(values)
    if finite_mask.all():
        return values
    finite_values = values[finite_mask]
    if finite_values.numel() == 0:
        return torch.zeros_like(values)
    high = finite_values.max().item()
    low  = finite_values.min().item()
    return torch.nan_to_num(values, nan=high, posinf=high, neginf=low)


def KL_neighbor_loss(predictions, targets, mask_len):
    x1 = predictions.squeeze().cpu().detach().float()
    x2 = targets.squeeze().cpu().detach().float()
    mean_x1 = x1.mean(0)
    mean_x2 = x2.mean(0)
    nn_nodes = x1.shape[0]
    h_dim    = x1.shape[1]
    cov_x1 = (x1 - mean_x1).T.matmul(x1 - mean_x1) / max(nn_nodes - 1, 1)
    cov_x2 = (x2 - mean_x2).T.matmul(x2 - mean_x2) / max(nn_nodes - 1, 1)
    eye    = torch.eye(h_dim, dtype=cov_x1.dtype, device=cov_x1.device)
    cov_x1 = cov_x1 + eye + 1e-6 * eye
    cov_x2 = cov_x2 + eye + 1e-6 * eye
    _, logdet1 = torch.linalg.slogdet(cov_x1)
    _, logdet2 = torch.linalg.slogdet(cov_x2)
    cov_x2_inv  = torch.linalg.pinv(cov_x2)
    mean_diff   = (mean_x2 - mean_x1).reshape(1, -1)
    KL_loss = 0.5 * ((logdet1 - logdet2) - h_dim
                     + torch.trace(cov_x2_inv.matmul(cov_x1))
                     + mean_diff.matmul(cov_x2_inv).matmul(mean_diff.T).squeeze())
    KL_loss = torch.nan_to_num(KL_loss, nan=0.0, posinf=1e6, neginf=0.0).to(device)
    return KL_loss

## Layer Definitions

In [4]:
class MLP(nn.Module):
    def __init__(self, num_layers, input_dim, hidden_dim, output_dim):
        super(MLP, self).__init__()
        self.linear_or_not = True
        self.num_layers = num_layers
        if num_layers < 1:
            raise ValueError('num_layers must be >= 1')
        elif num_layers == 1:
            self.linear = nn.Linear(input_dim, output_dim)
        else:
            self.linear_or_not = False
            self.linears    = nn.ModuleList()
            self.batch_norms = nn.ModuleList()
            self.linears.append(nn.Linear(input_dim, hidden_dim))
            for _ in range(num_layers - 2):
                self.linears.append(nn.Linear(hidden_dim, hidden_dim))
            self.linears.append(nn.Linear(hidden_dim, output_dim))
            for _ in range(num_layers - 1):
                self.batch_norms.append(nn.BatchNorm1d(hidden_dim))

    def forward(self, x):
        if self.linear_or_not:
            return self.linear(x)
        h = x
        for layer in range(self.num_layers - 1):
            h = self.linears[layer](h)
            if len(h.shape) > 2:
                h = h.transpose(0, 1).transpose(1, 2)
            h = self.batch_norms[layer](h)
            if len(h.shape) > 2:
                h = h.transpose(1, 2).transpose(0, 1)
            h = F.relu(h)
        return self.linears[self.num_layers - 1](h)


class MLP_generator(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(MLP_generator, self).__init__()
        self.linear  = nn.Linear(input_dim, output_dim)
        self.linear2 = nn.Linear(output_dim, output_dim)
        self.linear3 = nn.Linear(output_dim, output_dim)
        self.linear4 = nn.Linear(output_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.linear(x))
        x = F.relu(self.linear2(x))
        x = F.relu(self.linear3(x))
        return self.linear4(x)


class PairNorm(nn.Module):
    def __init__(self, mode='PN', scale=10):
        assert mode in ['None', 'PN', 'PN-SI', 'PN-SCS']
        super(PairNorm, self).__init__()
        self.mode  = mode
        self.scale = scale

    def forward(self, x):
        if self.mode == 'None':
            return x
        col_mean = x.mean(dim=0)
        if self.mode == 'PN':
            x = x - col_mean
            rownorm_mean = (1e-6 + x.pow(2).sum(dim=1).mean()).sqrt()
            x = self.scale * x / rownorm_mean
        if self.mode == 'PN-SI':
            x = x - col_mean
            rownorm_individual = (1e-6 + x.pow(2).sum(dim=1, keepdim=True)).sqrt()
            x = self.scale * x / rownorm_individual
        if self.mode == 'PN-SCS':
            rownorm_individual = (1e-6 + x.pow(2).sum(dim=1, keepdim=True)).sqrt()
            x = self.scale * x / rownorm_individual - col_mean
        return x


class FNN(nn.Module):
    def __init__(self, in_features, hidden, out_features, layer_num):
        super(FNN, self).__init__()
        self.linear1 = MLP(layer_num, in_features, hidden, out_features)
        self.linear2 = nn.Linear(out_features, out_features)

    def forward(self, x):
        x = self.linear1(x)
        x = self.linear2(F.relu(x))
        return F.relu(x)

## GAD-NR Model (GNNStructEncoder)

In [5]:
class GNNStructEncoder(nn.Module):
    def __init__(self, in_dim0, in_dim, hidden_dim, layer_num, sample_size, device,
                 neighbor_num_list, GNN_name='GCN', norm_mode='PN-SCS', norm_scale=20,
                 lambda_loss1=0.001, lambda_loss2=0.8, lambda_loss3=0.5):
        super(GNNStructEncoder, self).__init__()

        self.mlp0          = nn.Linear(in_dim0, hidden_dim)
        self.norm          = PairNorm(norm_mode, norm_scale)
        self.out_dim       = hidden_dim
        self.lambda_loss1  = lambda_loss1   # λ_n : neighbor reconstruction weight
        self.lambda_loss2  = lambda_loss2   # λ_x : feature reconstruction weight
        self.lambda_loss3  = lambda_loss3   # λ_d : degree reconstruction weight

        if GNN_name == 'GIN':
            self.linear1      = MLP(layer_num, hidden_dim, hidden_dim, hidden_dim)
            self.graphconv1   = GINConv(self.linear1)
        elif GNN_name == 'GCN':
            self.graphconv1   = GCNConv(hidden_dim, hidden_dim)
        elif GNN_name == 'GAT':
            self.graphconv1   = GATConv(hidden_dim, hidden_dim)
        else:  # GraphSAGE
            self.graphconv1   = SAGEConv(hidden_dim, hidden_dim, aggr=args.aggregator)

        self.neighbor_num_list = neighbor_num_list
        self.tot_node          = len(neighbor_num_list)

        self.gaussian_mean = nn.Parameter(
            torch.FloatTensor(sample_size, hidden_dim).uniform_(-0.5/hidden_dim, 0.5/hidden_dim)).to(device)
        self.gaussian_log_sigma = nn.Parameter(
            torch.FloatTensor(sample_size, hidden_dim).uniform_(-0.5/hidden_dim, 0.5/hidden_dim)).to(device)

        self.m         = torch.distributions.Normal(
            torch.zeros(sample_size, hidden_dim), torch.ones(sample_size, hidden_dim))
        self.m_batched = torch.distributions.Normal(
            torch.zeros(sample_size, self.tot_node, hidden_dim),
            torch.ones(sample_size, self.tot_node, hidden_dim))

        self.mlp_gaussian_mean = nn.Parameter(
            torch.FloatTensor(hidden_dim).uniform_(-0.5/hidden_dim, 0.5/hidden_dim)).to(device)
        self.mlp_gaussian_log_sigma = nn.Parameter(
            torch.FloatTensor(hidden_dim).uniform_(-0.5/hidden_dim, 0.5/hidden_dim)).to(device)

        self.mlp_mean  = FNN(hidden_dim, hidden_dim, hidden_dim, 3)
        self.mlp_sigma = FNN(hidden_dim, hidden_dim, hidden_dim, 3)
        self.softplus  = nn.Softplus()

        self.mean_agg = SAGEConv(hidden_dim, hidden_dim, aggr=args.aggregator, normalize=False)
        self.std_agg  = PNAConv(hidden_dim, hidden_dim, aggregators=['std'],
                                scalers=['identity'], deg=neighbor_num_list)
        self.layer1_generator = MLP_generator(hidden_dim, hidden_dim)

        self.degree_decoder    = FNN(hidden_dim, hidden_dim, 1, 4)
        self.feature_decoder   = FNN(hidden_dim, hidden_dim, in_dim, 3)
        self.degree_loss_func  = nn.MSELoss()
        self.feature_loss_func = nn.MSELoss()
        self.in_dim            = in_dim
        self.sample_size       = sample_size
        self.init_projection   = FNN(in_dim, hidden_dim, hidden_dim, 1)

    def forward_encoder(self, x, edge_index):
        h0 = self.mlp0(x)
        l1 = self.graphconv1(h0, edge_index)
        return l1, h0

    def reconstruction_neighbors2(self, l1, h0, edge_index):
        mean_neigh = self.mean_agg(h0, edge_index).detach()
        std_neigh  = self.std_agg(h0, edge_index).detach()

        target_mean = mean_neigh
        target_cov  = torch.bmm(std_neigh.unsqueeze(-1), std_neigh.unsqueeze(1))

        self_embedding  = l1.unsqueeze(0).repeat(self.sample_size, 1, 1)
        generated_mean  = self.mlp_mean(self_embedding)
        generated_sigma = self.mlp_sigma(self_embedding)

        std_z = self.m_batched.sample().to(device)
        var   = generated_mean + generated_sigma.exp() * std_z
        nhij  = self.layer1_generator(var)

        gen_mean = torch.mean(nhij, dim=0)
        gen_std  = torch.std(nhij, dim=0)
        gen_cov  = torch.bmm(gen_std.unsqueeze(-1), gen_std.unsqueeze(1)) / self.sample_size

        tot_nodes = l1.shape[0]
        h_dim     = l1.shape[1]
        batch_eye = torch.eye(h_dim).to(device).unsqueeze(0).repeat(tot_nodes, 1, 1)

        target_cov = target_cov + batch_eye + 1e-6 * batch_eye
        gen_cov    = gen_cov    + batch_eye + 1e-6 * batch_eye

        _, logdet_target = torch.linalg.slogdet(target_cov)
        _, logdet_gen    = torch.linalg.slogdet(gen_cov)
        inv_gen_cov      = torch.linalg.pinv(gen_cov)
        trace_mat        = torch.matmul(inv_gen_cov, target_cov)

        diff = gen_mean - target_mean
        z_   = torch.bmm(torch.bmm(diff.unsqueeze(1), inv_gen_cov), diff.unsqueeze(-1)).squeeze()

        KL_loss = 0.5 * (
            (logdet_target - logdet_gen)
            - h_dim
            + trace_mat.diagonal(offset=0, dim1=-1, dim2=-2).sum(-1)
            + z_
        )
        return torch.mean(KL_loss), KL_loss

    def neighbor_decoder(self, gij, gt_degree, h0, neighbor_dict, device, h, edge_index):
        tot_nodes    = gij.shape[0]
        degree_logits = F.relu(self.degree_decoder(gij))
        gt_degree_u   = gt_degree.unsqueeze(1)
        degree_loss   = self.degree_loss_func(degree_logits, gt_degree_u.float())
        degree_loss_per_node = (degree_logits - gt_degree_u).pow(2)

        h_loss = 0.0
        feature_loss = 0.0
        loss_list      = []
        loss_per_list  = []
        feat_loss_list = []

        # Sample 3 times to reduce noise (mirrors original code)
        for _ in range(3):
            h0_prime = self.feature_decoder(gij)
            feat_per_node = (h0 - h0_prime).pow(2).mean(1)
            feat_loss_list.append(feat_per_node)

            local_loss, local_loss_per = self.reconstruction_neighbors2(gij, h0, edge_index)
            loss_list.append(local_loss)
            loss_per_list.append(local_loss_per)

        loss_stack     = torch.stack(loss_list)
        h_loss        += torch.mean(loss_stack)
        h_loss_per     = torch.mean(torch.stack(loss_per_list), dim=0).reshape(tot_nodes, 1)

        feat_loss_per  = torch.mean(torch.stack(feat_loss_list), dim=0).reshape(tot_nodes, 1)
        feature_loss  += torch.mean(torch.stack(feat_loss_list))

        degree_loss_per_node = degree_loss_per_node.reshape(tot_nodes, 1)

        # Training loss (Eq. 7): uses lambda_loss1=λ_n, lambda_loss2=λ_x, lambda_loss3=λ_d
        loss = (self.lambda_loss1 * h_loss
                + self.lambda_loss3 * degree_loss
                + self.lambda_loss2 * feature_loss)
        loss_per_node = (self.lambda_loss1 * h_loss_per
                         + self.lambda_loss3 * degree_loss_per_node
                         + self.lambda_loss2 * feat_loss_per)

        return loss, loss_per_node, h_loss_per, degree_loss_per_node, feat_loss_per

    def forward(self, edge_index, x, gt_degree, neighbor_dict, device):
        l1, h0 = self.forward_encoder(x, edge_index)
        return self.neighbor_decoder(l1, gt_degree, h0, neighbor_dict, device, x, edge_index)

## Training Function

**Critical fix**: Lambda scheduling is restored inside the epoch loop.
Every `loss_step` epochs: `lambda_loss2 (λ_x) += 0.5` and `lambda_loss3 (λ_d) /= 2`.
This progressive shift toward feature reconstruction is essential for good contextual anomaly detection.
With `loss_step=100` and 500 epochs, the model undergoes 5 scheduling steps.

In [6]:
def train(data, y, yc, ys, yj, ysj, lr, epoch, device, encoder,
          lambda_loss1, lambda_loss2, lambda_loss3, hidden_dim,
          sample_size=10, loss_step=100, real_loss=False,
          calculate_contextual=False, calculate_structural=False):
    """
    lambda_loss1 = lambda_n  : neighbor reconstruction weight (paper: 0.001)
    lambda_loss2 = lambda_x  : feature reconstruction weight  (paper: 0.8)
    lambda_loss3 = lambda_d  : degree reconstruction weight   (paper: 0.5)

    loss_step: every this many epochs, lambda_loss2 += 0.5 and lambda_loss3 /= 2
               Use 100 (original default). Setting to >epoch disables scheduling.
    """

    in_nodes  = data.edge_index[0, :]
    out_nodes = data.edge_index[1, :]

    neighbor_dict = {}
    for src, dst in zip(in_nodes.tolist(), out_nodes.tolist()):
        if src not in neighbor_dict:
            neighbor_dict[src] = []
        neighbor_dict[src].append(dst)

    neighbor_num_list = torch.tensor(
        [len(neighbor_dict[i]) for i in range(len(neighbor_dict))]
    ).to(device)

    in_dim = data.x.shape[1]
    GNNModel = GNNStructEncoder(
        in_dim, hidden_dim, hidden_dim, 2, sample_size,
        device=device, neighbor_num_list=neighbor_num_list,
        GNN_name=encoder,
        lambda_loss1=lambda_loss1,
        lambda_loss2=lambda_loss2,
        lambda_loss3=lambda_loss3)
    GNNModel.to(device)

    degree_param_ids = set(map(id, GNNModel.degree_decoder.parameters()))
    base_params      = [p for p in GNNModel.parameters() if id(p) not in degree_param_ids]
    opt = torch.optim.Adam(
        [{'params': base_params},
         {'params': GNNModel.degree_decoder.parameters(), 'lr': 1e-2}],
        lr=lr, weight_decay=0.0003)

    best_auc_benchmark        = 0.0
    best_auc_contextual       = 0.0
    best_auc_structural       = 0.0
    best_auc_joint            = 0.0
    best_auc_structural_joint = 0.0
    epoch_times               = []

    def safe_normalize(t):
        rng = torch.max(t) - torch.min(t)
        return t / rng if rng > 1e-12 else torch.zeros_like(t)

    for i in tqdm(range(1, epoch + 1), desc='Epochs', leave=False):
        t_start = time.time()

        # ===================================================================
        # CRITICAL FIX: Lambda scheduling (was MISSING in prior attempt)
        # Progressively increases feature loss weight and decreases degree
        # loss weight over training.  loss_step=100 => 5 updates in 500 epochs
        # ===================================================================
        if i % loss_step == 0:
            GNNModel.lambda_loss2 = GNNModel.lambda_loss2 + 0.5
            GNNModel.lambda_loss3 = GNNModel.lambda_loss3 / 2

        loss, loss_per_node, h_loss, degree_loss, feature_loss = GNNModel(
            data.edge_index, data.x, neighbor_num_list, neighbor_dict, device=device)

        if not torch.isfinite(loss):
            print(f'Non-finite loss at epoch {i}, stopping early.')
            break

        loss_pn  = sanitize_score_tensor(loss_per_node.cpu().detach())
        h_d      = h_loss.cpu().detach()
        deg_d    = degree_loss.cpu().detach()
        feat_d   = feature_loss.cpu().detach()

        h_norm    = safe_normalize(h_d)
        deg_norm  = safe_normalize(deg_d)
        feat_norm = safe_normalize(feat_d)

        # Anomaly score (Eq. 8): adaptive combination using lambda-prime weights
        comb_loss = (args.h_loss_weight     * h_norm
                     + args.degree_loss_weight  * deg_norm
                     + args.feature_loss_weight * feat_norm)
        comp_loss = loss_pn if real_loss else sanitize_score_tensor(comb_loss)

        try:
            auc = eval_roc_auc(y.numpy(), comp_loss.numpy()) * 100
            best_auc_benchmark = max(best_auc_benchmark, auc)
        except Exception:
            pass

        if calculate_contextual and isinstance(yc, torch.Tensor) and yc.sum() > 0:
            try:
                c_auc = eval_roc_auc(yc.numpy(), comp_loss.numpy()) * 100
                best_auc_contextual = max(best_auc_contextual, c_auc)
            except Exception:
                pass

        if calculate_structural:
            if isinstance(ys, torch.Tensor) and ys.sum() > 0:
                try:
                    s_auc = eval_roc_auc(ys.numpy(), comp_loss.numpy()) * 100
                    best_auc_structural = max(best_auc_structural, s_auc)
                except Exception:
                    pass
            if isinstance(yj, torch.Tensor) and yj.sum() > 0:
                try:
                    j_auc = eval_roc_auc(yj.numpy(), comp_loss.numpy()) * 100
                    best_auc_joint = max(best_auc_joint, j_auc)
                except Exception:
                    pass
            if isinstance(ysj, torch.Tensor) and ysj.sum() > 0:
                try:
                    sj_auc = eval_roc_auc(ysj.numpy(), comp_loss.numpy()) * 100
                    best_auc_structural_joint = max(best_auc_structural_joint, sj_auc)
                except Exception:
                    pass

        opt.zero_grad()
        loss.backward()
        opt.step()
        epoch_times.append(time.time() - t_start)

    return {
        'best_auc_benchmark':        best_auc_benchmark,
        'best_auc_contextual':       best_auc_contextual,
        'best_auc_structural':       best_auc_structural,
        'best_auc_joint':            best_auc_joint,
        'best_auc_structural_joint': best_auc_structural_joint,
        'avg_time_per_epoch':        float(np.mean(epoch_times)) if epoch_times else 0.0,
    }


def train_real_datasets(dataset_str, lambda_loss1, lambda_loss2, lambda_loss3,
                        epoch_num, lr, encoder, sample_size, loss_step, hidden_dim,
                        real_loss, calculate_contextual, calculate_structural,
                        contextual_n, contextual_k, structural_n, structural_m):
    try:
        data = load_data(dataset_str)
    except Exception:
        import urllib.request, zipfile
        url = f'https://github.com/pygod-team/data/raw/main/{dataset_str}.pt.zip'
        urllib.request.urlretrieve(url, f'{dataset_str}.pt.zip')
        with zipfile.ZipFile(f'{dataset_str}.pt.zip', 'r') as zf:
            zf.extractall('.')
        data = load_data(dataset_str)

    if args.normalize_feat:
        nf_min, nf_max = data.x.min(), data.x.max()
        data.x = (data.x - nf_min) / (nf_max + 1e-12)

    n_nodes = data.x.shape[0]
    yc = torch.zeros(n_nodes, dtype=torch.long)
    ys = torch.zeros(n_nodes, dtype=torch.long)
    yj = torch.zeros(n_nodes, dtype=torch.long)

    if calculate_contextual:
        data, yc = gen_contextual_outliers(data=data, n=contextual_n, k=contextual_k, random_state=42)
        yc = yc.cpu().detach()

    if calculate_structural:
        data, ys = gen_structural_outliers(data=data, n=structural_n, m=structural_m, p=0.2, random_state=42)
        ys = ys.cpu().detach()
        data, yj = gen_joint_structural_outliers(data=data, n=structural_n, m=structural_m, random_state=42)
        yj = yj.cpu().detach()

    ysj = torch.logical_or(ys, yj).int()
    y   = data.y.bool().cpu().detach()

    # Add self-loops
    edge_index = data.edge_index.cpu()
    self_loops = torch.tensor([list(range(n_nodes)), list(range(n_nodes))])
    data.edge_index = torch.cat([edge_index, self_loops], dim=1)
    data = data.to(device)

    return train(
        data, y, yc, ys, yj, ysj,
        lr=lr, epoch=epoch_num, device=device, encoder=encoder,
        lambda_loss1=lambda_loss1, lambda_loss2=lambda_loss2, lambda_loss3=lambda_loss3,
        hidden_dim=hidden_dim, sample_size=sample_size, loss_step=loss_step,
        real_loss=real_loss,
        calculate_contextual=calculate_contextual,
        calculate_structural=calculate_structural
    )

## Reproduce Tables 2, 3, and 4 — Weibo Dataset

### Hyperparameter mapping (code → paper notation)
| Argument | Paper symbol | Paper value (fixed) | Role |
|---|---|---|---|
| `lambda_loss1` | λ_n | **0.001** | Neighbor reconstruction weight |
| `lambda_loss2` | λ_x | **0.8** | Feature reconstruction weight |
| `lambda_loss3` | λ_d | **0.5** | Degree reconstruction weight |

### Ablation: zeroing out one component at a time
- *w/o feat. recon.*: set λ_x = 0 → `lambda_loss2 = 0`
- *w/o degree recon.*: set λ_d = 0 → `lambda_loss3 = 0`
- *w/o neighbor recon.*: set λ_n = 0 → `lambda_loss1 = 0`

In [7]:
ds  = 'weibo'
cfg = {'hidden_dim': 16, 'cn': 434, 'ck': 10, 'sn': 434, 'sm': 10}

# -----------------------------------------------------------------------
# CORRECTED ablation configs — paper values: λ_n=0.001, λ_x=0.8, λ_d=0.5
# Code mapping:  lambda_loss1=λ_n,  lambda_loss2=λ_x,  lambda_loss3=λ_d
# -----------------------------------------------------------------------
ABLATION_CONFIGS = [
    ('GAD-NR (w/o feat. recon.)',    {'l1': 0.001, 'l2': 0.0,  'l3': 0.5}),
    ('GAD-NR (w/o degree recon.)',   {'l1': 0.001, 'l2': 0.8,  'l3': 0.0}),
    ('GAD-NR (w/o neighbor recon.)', {'l1': 0.0,   'l2': 0.8,  'l3': 0.5}),
    ('GAD-NR',                       {'l1': 0.001, 'l2': 0.8,  'l3': 0.5}),
]

PAPER_REFS = {
    'Table2 (Benchmark)': {
        'GAD-NR (w/o feat. recon.)':    69.64,
        'GAD-NR (w/o degree recon.)':   70.09,
        'GAD-NR (w/o neighbor recon.)': 69.10,
        'GAD-NR':                       87.71,
    },
    'Table3 Contextual': {
        'GAD-NR (w/o feat. recon.)':    57.66,
        'GAD-NR (w/o degree recon.)':   64.28,
        'GAD-NR (w/o neighbor recon.)': 61.45,
        'GAD-NR':                       87.53,
    },
    'Table3 Struct+Joint': {
        'GAD-NR (w/o feat. recon.)':    50.81,
        'GAD-NR (w/o degree recon.)':   53.33,
        'GAD-NR (w/o neighbor recon.)': 50.29,
        'GAD-NR':                       62.35,
    },
}

results_dict = {}

for variant, lam in ABLATION_CONFIGS:
    print(f'\n{"="*70}')
    print(f'Running: {variant}')
    print(f'  lambda_loss1(λ_n)={lam["l1"]}, lambda_loss2(λ_x)={lam["l2"]}, lambda_loss3(λ_d)={lam["l3"]}')
    print(f'{"="*70}')

    random.seed(42); np.random.seed(42); torch.manual_seed(42)

    res = train_real_datasets(
        dataset_str=ds,
        lambda_loss1=lam['l1'],
        lambda_loss2=lam['l2'],
        lambda_loss3=lam['l3'],
        epoch_num=500,
        lr=0.01,
        encoder='GCN',
        sample_size=10,
        loss_step=100,   # CORRECTED: was 5000 (disabled scheduling); paper uses 100
        hidden_dim=cfg['hidden_dim'],
        real_loss=args.real_loss,
        calculate_contextual=True,
        calculate_structural=True,
        contextual_n=cfg['cn'], contextual_k=cfg['ck'],
        structural_n=cfg['sn'], structural_m=cfg['sm'],
    )
    results_dict[variant] = res
    print(f'  → Best Benchmark AUC: {res["best_auc_benchmark"]:.2f}%')
    print(f'  → Best Contextual AUC: {res["best_auc_contextual"]:.2f}%')
    print(f'  → Best Struct+Joint AUC: {res["best_auc_structural_joint"]:.2f}%')
    print(f'  → Avg time/epoch: {res["avg_time_per_epoch"]:.4f}s')

print('\nAll variants done.')


Running: GAD-NR (w/o feat. recon.)
  lambda_loss1(λ_n)=0.001, lambda_loss2(λ_x)=0.0, lambda_loss3(λ_d)=0.5


  → Best Benchmark AUC: 80.72%
  → Best Contextual AUC: 82.66%
  → Best Struct+Joint AUC: 68.45%
  → Avg time/epoch: 0.1512s

Running: GAD-NR (w/o degree recon.)
  lambda_loss1(λ_n)=0.001, lambda_loss2(λ_x)=0.8, lambda_loss3(λ_d)=0.0


  → Best Benchmark AUC: 46.90%
  → Best Contextual AUC: 77.69%
  → Best Struct+Joint AUC: 65.46%
  → Avg time/epoch: 0.1153s

Running: GAD-NR (w/o neighbor recon.)
  lambda_loss1(λ_n)=0.0, lambda_loss2(λ_x)=0.8, lambda_loss3(λ_d)=0.5


  → Best Benchmark AUC: 80.11%
  → Best Contextual AUC: 84.38%
  → Best Struct+Joint AUC: 69.56%
  → Avg time/epoch: 0.1109s

Running: GAD-NR
  lambda_loss1(λ_n)=0.001, lambda_loss2(λ_x)=0.8, lambda_loss3(λ_d)=0.5


  → Best Benchmark AUC: 81.14%
  → Best Contextual AUC: 85.40%
  → Best Struct+Joint AUC: 68.63%
  → Avg time/epoch: 0.1112s

All variants done.


## Display Results — Tables 2, 3, and 4

In [8]:
# ---- Table 2 & 3 -------------------------------------------------------
rows = []
for variant, _ in ABLATION_CONFIGS:
    r = results_dict[variant]
    rows.append({
        'Model':           variant,
        'Table':           'Table 2 — Benchmark',
        'Reproduced AUC':  round(r['best_auc_benchmark'], 2),
        'Paper AUC':       PAPER_REFS['Table2 (Benchmark)'][variant],
    })
    rows.append({
        'Model':           variant,
        'Table':           'Table 3 — Contextual',
        'Reproduced AUC':  round(r['best_auc_contextual'], 2),
        'Paper AUC':       PAPER_REFS['Table3 Contextual'][variant],
    })
    rows.append({
        'Model':           variant,
        'Table':           'Table 3 — Struct+Joint',
        'Reproduced AUC':  round(r['best_auc_structural_joint'], 2),
        'Paper AUC':       PAPER_REFS['Table3 Struct+Joint'][variant],
    })

df = pd.DataFrame(rows)[['Table', 'Model', 'Paper AUC', 'Reproduced AUC']]
df['Delta'] = df['Reproduced AUC'] - df['Paper AUC']

print('\n=== Tables 2 & 3 — Weibo (single run) ===')
print(df.to_string(index=False))

# ---- Table 4 -----------------------------------------------------------
gadnr_res = results_dict['GAD-NR']
df4 = pd.DataFrame([
    {
        'Algorithm':                  'GAD-NR',
        'Paper AUC':                  89.71,
        'Reproduced AUC':             round(gadnr_res['best_auc_benchmark'], 2),
        'Paper s/epoch':              0.544,
        'Reproduced s/epoch':         round(gadnr_res['avg_time_per_epoch'], 4),
    },
    {
        'Algorithm':                  'NWR-GAE (paper only, no public code)',
        'Paper AUC':                  73.68,
        'Reproduced AUC':             'N/A',
        'Paper s/epoch':              48.312,
        'Reproduced s/epoch':         'N/A',
    },
])
print('\n=== Table 4 — NWR-GAE vs GAD-NR (Weibo) ===')
print(df4.to_string(index=False))


=== Tables 2 & 3 — Weibo (single run) ===
                 Table                        Model  Paper AUC  Reproduced AUC  Delta
   Table 2 — Benchmark    GAD-NR (w/o feat. recon.)      69.64           80.72  11.08
  Table 3 — Contextual    GAD-NR (w/o feat. recon.)      57.66           82.66  25.00
Table 3 — Struct+Joint    GAD-NR (w/o feat. recon.)      50.81           68.45  17.64
   Table 2 — Benchmark   GAD-NR (w/o degree recon.)      70.09           46.90 -23.19
  Table 3 — Contextual   GAD-NR (w/o degree recon.)      64.28           77.69  13.41
Table 3 — Struct+Joint   GAD-NR (w/o degree recon.)      53.33           65.46  12.13
   Table 2 — Benchmark GAD-NR (w/o neighbor recon.)      69.10           80.11  11.01
  Table 3 — Contextual GAD-NR (w/o neighbor recon.)      61.45           84.38  22.93
Table 3 — Struct+Joint GAD-NR (w/o neighbor recon.)      50.29           69.56  19.27
   Table 2 — Benchmark                       GAD-NR      87.71           81.14  -6.57
  Table 3 —